In [ ]:
from pathlib import Path 
import matplotlib.pyplot as plt
import rioxarray as rxr 
import sys
sys.path.append("..") #crawl up one level to project root (notebook runs from figures/)
from utils.paths import load_paths
paths=load_paths()

fsc_gt_p=paths["data"]["fsc_gt"] / "VALIDATION_fsc_20250604_cloudmasked.tif"

fsc_pred_p=Path("../model/FSC_PRED_20250606.tif")


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

fsc_gt_path = paths["data"]["fsc_gt"] / "VALIDATION_fsc_20250604_cloudmasked.tif"
fsc_pred_path = Path("../model/FSC_PRED_20250606.tif")

# ------------------------------------------------------------------
# Load rasters
# ------------------------------------------------------------------

gt = rxr.open_rasterio(fsc_gt_path, mask=True).squeeze()
pred = rxr.open_rasterio(fsc_pred_path, mask=True).squeeze()

# ------------------------------------------------------------------
# Align prediction raster to ground truth grid
# ------------------------------------------------------------------

pred_aligned = pred.rio.reproject_match(gt)

# ------------------------------------------------------------------
# Convert to numpy arrays
# ------------------------------------------------------------------

gt_values = gt.values.flatten()
pred_values = pred_aligned.values.flatten()

# ------------------------------------------------------------------
# Filter valid FSC values (0–1)
# ------------------------------------------------------------------

valid_mask = (
    (gt_values >= 0) & (gt_values <= 1) &
    (pred_values >= 0) & (pred_values <= 1)
)

gt_valid = gt_values[valid_mask]
pred_valid = pred_values[valid_mask]

# ------------------------------------------------------------------
# Calculate evaluation metrics
# ------------------------------------------------------------------

rmse = np.sqrt(np.mean((pred_valid - gt_valid) ** 2))
bias = np.mean(pred_valid - gt_valid)
r = np.corrcoef(gt_valid, pred_valid)[0, 1]

# ------------------------------------------------------------------
# Plot hexbin density scatter
# ------------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(7, 7))

hb = ax.hexbin(
    gt_valid,
    pred_valid,
    gridsize=55,
    cmap="viridis",
    mincnt=1
)

# Perfect prediction line
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=2, color="red")

# Axis formatting
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")

ax.set_xlabel("Ground Truth FSC")
ax.set_ylabel("Predicted FSC")
ax.set_title("Predicted vs Ground Truth FSC")

# Metrics box
metrics_text = (
    f"R = {r:.3f}\n"
    f"RMSE = {rmse:.3f}\n"
    f"Bias = {bias:.3f}\n"
    f"N = {len(gt_valid):,}"
)

ax.text(
    0.05,
    0.95,
    metrics_text,
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
)

# Colorbar
cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("Pixel count")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker
import rasterio.enums
import rioxarray as rxr

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

fsc_gt_path = paths["data"]["fsc_gt"] / "VALIDATION_fsc_20250604_cloudmasked.tif"
fsc_pred_path = Path("../model/FSC_PRED_20250606.tif")

# ------------------------------------------------------------------
# Load & align rasters
# ------------------------------------------------------------------

gt   = rxr.open_rasterio(fsc_gt_path,   masked=True).squeeze()
pred = rxr.open_rasterio(fsc_pred_path, masked=True).squeeze()

pred_aligned = pred.rio.reproject_match(gt, resampling=rasterio.enums.Resampling.nearest)

# ------------------------------------------------------------------
# Flatten & mask
# ------------------------------------------------------------------

gt_values   = gt.values.flatten()
pred_values = pred_aligned.values.flatten()

valid_mask = (
    ~np.isnan(gt_values) & ~np.isnan(pred_values) &
    (gt_values   >= 0) & (gt_values   <= 1) &
    (pred_values >= 0) & (pred_values <= 1)
)

gt_valid   = gt_values[valid_mask]
pred_valid = pred_values[valid_mask]

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------

rmse = np.sqrt(np.mean((pred_valid - gt_valid) ** 2))
bias = np.mean(pred_valid - gt_valid)
r    = np.corrcoef(gt_valid, pred_valid)[0, 1]
r2   = r ** 2

# ------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------

plt.rcParams.update({
    "font.family":  "sans-serif",
    "font.size":    14,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "axes.titleweight": "bold",
})

fig, ax = plt.subplots(figsize=(7, 7), facecolor="white")
ax.set_facecolor("#e4e4e4")

# Hexbin
hb = ax.hexbin(
    gt_valid, pred_valid,
    gridsize=60,
    cmap="plasma",
    mincnt=1,
    norm=mcolors.LogNorm(),
    linewidths=0.2,
)

# 1:1 line
ax.plot([0, 1], [0, 1], "-", linewidth=4, color="#000000", label="1:1 line", zorder=5)

# Axes
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_xlabel("Ground Truth FSC", labelpad=10)
ax.set_ylabel("Predicted FSC",    labelpad=10)
ax.set_title("Predicted vs. Ground Truth FSC\nSupport Vector Regressor (B2, B3, B12)", pad=14)

# Grid
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5, color="grey")
ax.set_axisbelow(True)

# Metrics box
metrics_text = (
    f"$R^2$   =  {r2:.2f}\n"
    f"RMSE =  {rmse:.1f}\n"
    f"N    =  {len(gt_valid):}"
)

ax.text(
    0.05, 0.97, metrics_text,
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=13,
    fontfamily="monospace",
    bbox=dict(boxstyle="round,pad=0.6", facecolor="white", edgecolor="#cccccc", alpha=0.95),
)

# Colorbar
cbar = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
cbar.set_label("Pixel count (log scale)", fontsize=15)
cbar.ax.tick_params(labelsize=11)

plt.tight_layout()
Path("../results/plots").mkdir(parents=True, exist_ok=True)
plt.savefig("../results/plots/fsc_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# absolute error
error = np.abs(pred_valid - gt_valid)

# create dataframe
df = pd.DataFrame({
    "gt": gt_valid,
    "pred": pred_valid,
    "error": error
})

# bin by ground truth FSC
bins = np.linspace(0, 1, 11)
df["bin"] = pd.cut(df["gt"], bins)

# compute statistics
rmse_by_bin = df.groupby("bin").apply(lambda x: np.sqrt(np.mean((x["pred"]-x["gt"])**2)))
mae_by_bin = df.groupby("bin")["error"].mean()
bin_centers = (bins[:-1] + bins[1:]) / 2

# plot
plt.figure(figsize=(7,5))

plt.plot(bin_centers, rmse_by_bin, marker="o", label="RMSE")
plt.plot(bin_centers, mae_by_bin, marker="s", label="MAE")

plt.xlabel("Ground Truth FSC")
plt.ylabel("Error")
plt.title("Prediction Error vs Snow Fraction")

plt.grid(alpha=0.3)
plt.legend()

plt.show()